# RAWG Pipeline: PySpark Notebook

This notebook takes your existing `transform.py` script and breaks it into cells with explanations, so you can run it step-by-step.

Each section has a markdown cell explaining *what* the code does and *why*, plus a callout on how the same job would look inside a **Fabric notebook**.

## 1. Imports

Nothing unusual here: `duckdb` for reading your bronze tables, `pyspark.sql.SparkSession` to start a Spark job, `functions as F` for column expressions, and `types` for schema definitions.

**Fabric tie-in:** in a Fabric notebook you'd skip `duckdb` entirely and read straight from a **Lakehouse** table using Spark SQL or `spark.read.format("delta")`. You also don't need to build a `SparkSession` yourself — Fabric notebooks give you one already instantiated as the variable `spark`. That's a detail worth remembering for the exam: candidates sometimes get asked why you don't see `SparkSession.builder` in a Fabric notebook.


In [7]:
import os

import duckdb
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    FloatType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)


## 2. Paths

These just resolve the output directory (where Parquet files land) and the DuckDB file (your bronze layer). Nothing Spark-specific, just filesystem bookkeeping.

**Fabric tie-in:** in Fabric, "paths" become **OneLake** paths, e.g. `abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>.Lakehouse/Tables/games`. OneLake is the single logical data lake behind every workspace — one copy of the data, referenced the same way whether you're in a notebook, a Warehouse, or a Power BI semantic model. OneLake is the "OneDrive for data" analogy Microsoft uses.


In [8]:
OUTPUT_DIR = os.path.abspath(
    os.path.join(os.path.dirname("__file__"), "..", "..", "data", "spark")
)

DB_PATH = os.path.abspath(
    os.path.join(os.path.dirname("__file__"), "..", "..", "rawg_data.duckdb")
)


## 3. Schemas

`StructType`/`StructField` define an explicit schema for the JSON you're about to parse out of the bronze `raw_json` column. Explicit schemas are good practice because Spark won't have to infer types by scanning the data (slow, and sometimes wrong), and your pipeline fails fast if the source shape changes.

**Fabric tie-in:** this is directly relevant to **schema enforcement vs schema evolution** in Delta Lake. Delta tables enforce schema on write by default (a mismatched column throws an error); you have to explicitly opt in to schema evolution (`mergeSchema` option, or `ALTER TABLE`) if you want new columns to be accepted automatically. Defining your schema up front here is the same instinct as Delta's enforce-by-default behaviour.


In [9]:
GAME_SCHEMA = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("rating", FloatType(), True),
    StructField("ratings_count", IntegerType(), True),
    StructField("released", StringType(), True),
])

GENRE_SCHEMA = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("slug", StringType(), True),
])

PLATFORM_SCHEMA = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("slug", StringType(), True),
])


## 4. Starting Spark

`SparkSession.builder` with `.master("local[*]")` spins up a local Spark cluster using all available cores on your machine. This is what you need for a standalone script.

**Fabric tie-in:** in a Fabric notebook, delete this cell entirely — `spark` already exists.

In [10]:
def build_spark() -> SparkSession:
    """Create and return a local SparkSession."""
    return (
        SparkSession.builder
        .appName("rawg_pipeline")
        .master("local[*]")
        .getOrCreate()
    )


## 5. Reading bronze data

This function opens a read-only DuckDB connection, runs a SQL `SELECT *`, pulls the result into a pandas DataFrame, then hands that to Spark via `spark.createDataFrame()`. It's a pragmatic bridge: DuckDB is your bronze store, and this avoids setting up a JDBC driver just to get data into Spark.

**Fabric tie-in:** this whole function is the piece that changes most in Fabric. Bronze data would already sit as a **Delta table in a Lakehouse**, so you'd replace this with something like:

```python
df = spark.read.format("delta").load("Tables/bronze_games")
# or, using the Lakehouse SQL endpoint:
df = spark.sql("SELECT * FROM bronze_lakehouse.bronze_games")
```

No pandas bridge, no DuckDB — Spark reads Delta natively. 


In [11]:
def read_bronze(spark: SparkSession, table: str):
    """
    Read a bronze DuckDB table into a Spark DataFrame via pandas bridge.
    Avoids JDBC/Java dependency by using the native duckdb Python client.
    """
    conn = duckdb.connect(DB_PATH, read_only=True)
    pandas_df = conn.execute(f"SELECT * FROM {table}").df()
    conn.close()
    return spark.createDataFrame(pandas_df)


## 6. Transforming games

Walking through the chain:
- `F.from_json(F.col("raw_json"), GAME_SCHEMA)` parses the JSON string column into a structured column called `data`, typed according to `GAME_SCHEMA`.
- `.select(F.col("data.id")...)` pulls fields back out of that nested struct and renames them with `.alias(...)`.
- `F.to_date(F.col("data.released"), "yyyy-MM-dd")` converts the released-date string into an actual `DateType`, using a format string. Worth knowing: Spark's `to_date` format patterns follow Java's `DateTimeFormatter`, not Python's `strftime` — `yyyy-MM-dd`, not `%Y-%m-%d`.
- `.dropDuplicates(["rawg_id"])` deduplicates on the natural key.
- `parsed.write.mode("overwrite").parquet(output_path)` writes the result as Parquet, replacing whatever was there before.

**Fabric tie-in:**
1. `.write.mode("overwrite").parquet(...)` → in Fabric you'd almost always write `.format("delta")` instead of `.parquet`. Delta gives you ACID transactions, time travel, and is what makes a table queryable by the SQL endpoint and usable in Direct Lake mode. Plain Parquet files dropped into a Lakehouse's Files section are *not* automatically registered as queryable tables.
2. `write.mode("overwrite")` fully replaces the table. Compare this against `.mode("append")` and against a proper `MERGE` (upsert).


In [12]:
def transform_games(spark: SparkSession) -> None:
    """Read bronze games, parse JSON, type fields, write Parquet."""
    df = read_bronze(spark, "bronze.bronze_games")

    parsed = df.select(
        F.from_json(F.col("raw_json"), GAME_SCHEMA).alias("data")
    ).select(
        F.col("data.id").alias("rawg_id"),
        F.col("data.name").alias("name"),
        F.col("data.rating").alias("rating"),
        F.col("data.ratings_count").alias("ratings_count"),
        F.to_date(F.col("data.released"), "yyyy-MM-dd").alias("released"),
    ).dropDuplicates(["rawg_id"])

    output_path = os.path.join(OUTPUT_DIR, "games")
    parsed.write.mode("overwrite").parquet(output_path)
    print(f"Written {parsed.count()} game records to {output_path}")


## 7. Transforming genres and platforms

Same pattern as `transform_games`, just against the smaller reference-data schemas. Repeating the pattern like this is a good moment to notice you could refactor it into one generic function parameterised by table name and schema.


In [13]:
def transform_genres(spark: SparkSession) -> None:
    """Read bronze genres, parse JSON, write Parquet."""
    df = read_bronze(spark, "bronze.bronze_genres")

    parsed = df.select(
        F.from_json(F.col("raw_json"), GENRE_SCHEMA).alias("data")
    ).select(
        F.col("data.id").alias("rawg_id"),
        F.col("data.name").alias("name"),
        F.col("data.slug").alias("slug"),
    ).dropDuplicates(["rawg_id"])

    output_path = os.path.join(OUTPUT_DIR, "genres")
    parsed.write.mode("overwrite").parquet(output_path)
    print(f"Written {parsed.count()} genre records to {output_path}")


In [14]:
def transform_platforms(spark: SparkSession) -> None:
    """Read bronze platforms, parse JSON, write Parquet."""
    df = read_bronze(spark, "bronze.bronze_platforms")

    parsed = df.select(
        F.from_json(F.col("raw_json"), PLATFORM_SCHEMA).alias("data")
    ).select(
        F.col("data.id").alias("rawg_id"),
        F.col("data.name").alias("name"),
        F.col("data.slug").alias("slug"),
    ).dropDuplicates(["rawg_id"])

    output_path = os.path.join(OUTPUT_DIR, "platforms")
    parsed.write.mode("overwrite").parquet(output_path)
    print(f"Written {parsed.count()} platform records to {output_path}")


## 8. Running the pipeline

In the original script this sat behind `if __name__ == "__main__":`, which only fires when you run the file directly (`python transform.py`), not when it's imported. In a notebook there's no equivalent — cells just run top to bottom — so it's a plain call instead.

Run this cell last, once the functions above have all been defined and executed.


In [15]:
spark = build_spark()
print("Running PySpark bronze -> Parquet transforms...")
transform_games(spark)
transform_genres(spark)
transform_platforms(spark)
spark.stop()
print("Done.")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/24 12:29:10 WARN Utils: Your hostname, codespaces-697a43, resolves to a loopback address: 127.0.0.1; using 10.0.0.37 instead (on interface eth0)
26/07/24 12:29:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/24 12:29:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Running PySpark bronze -> Parquet transforms...


26/07/24 12:29:30 WARN TaskSetManager: Stage 0 contains a task of very large size (4980 KiB). The maximum recommended task size is 1000 KiB.
26/07/24 12:29:36 WARN TaskSetManager: Stage 3 contains a task of very large size (4980 KiB). The maximum recommended task size is 1000 KiB.


Written 1000 game records to /workspaces/data-engineering-portfolio/projects/rawg_pipeline/data/spark/games
Written 19 genre records to /workspaces/data-engineering-portfolio/projects/rawg_pipeline/data/spark/genres
Written 51 platform records to /workspaces/data-engineering-portfolio/projects/rawg_pipeline/data/spark/platforms
Done.


## Quick Fabric recap from this notebook

- **Schema enforcement vs evolution** — explicit `StructType` schemas here mirror Delta's enforce-on-write default; evolution needs an explicit opt-in.
- **OneLake** — one logical copy of data, referenced by the same path regardless of which Fabric engine (Spark, SQL endpoint, Power BI) touches it.
- **Fabric-native Spark session** — `spark` is pre-built in a Fabric notebook; you never call `SparkSession.builder` there.
- **Delta vs Parquet writes** — `.format("delta")` gets you ACID transactions, time travel, and Direct Lake compatibility; plain `.parquet` doesn't register as a queryable table.
- **Write modes** — `overwrite` (full reload) vs `append` (incremental) vs `MERGE` (upsert).
